# Exploit Text Graph — End-to-End Demo

Runs **both research thrusts** from Samtani's NSF CAREER proposal (pages 9–14) in one notebook:

- **RT1**: build per-spell Exploit Text Graphs → train Diachronic Graph Transformer → detect semantic shifts → forecast with ARIMA → compare to word2vec + naive baselines.
- **RT2**: pretrain Context Transformer Encoder with masked + contrastive objectives → fine-tune on EV pairs → rank → compare to TF-IDF, BM25, LSA, and optional SimCSE / Contriever.

Default is `Config.tiny()` — full end-to-end run in ~2 minutes on MPS/CPU, useful for sanity-checking the pipeline. Swap to `Config.smoke()` for a richer but still-laptop-friendly run, or `Config.full()` for a cluster-scale run.

## 1. Setup

In [1]:
from __future__ import annotations

import os, sys, json, warnings, importlib, time
warnings.filterwarnings('ignore')

# Ensure notebook runs from repo root (so caches land in ./cache).
HERE = os.getcwd()
REPO = os.path.abspath(os.path.join(HERE, '..'))
os.chdir(REPO)
print('working dir:', os.getcwd())

from etg.config import Config
from etg.device import auto_device, device_summary
from etg.seeding import set_global_seed
from etg.logging_utils import get_logger

# Pick the run scale:
#   Config.tiny()  — sanity-check, finishes in < 2 min on MPS/CPU.
#   Config.smoke() — richer laptop-scale run (~10-20 min on MPS).
#   Config.full()  — cluster-scale run.
config = Config.smoke()

# Set True to include SimCSE / Contriever baselines (downloads ~500MB from HuggingFace).
RUN_PRETRAINED_BASELINES = False

config.ensure_dirs()
set_global_seed(config.seed)
device = auto_device()
log = get_logger('etg.notebook')
log.info(f'device = {device_summary(device)}')
log.info(f'config hash = {config.hash()}')
log.info(f'cache dir = {config.cache_dir}')
_T0 = time.time()

results: dict = {'config_hash': config.hash(), 'rt1': {}, 'rt2': {}}

working dir: /Volumes/AppSupportSSD/Exploit Text Graph/exploit-text-graph


[04/14/26 22:08:26] INFO     device = MPS (Apple Silicon)

                    INFO     config hash = a95acf010aa5ce0c

                    INFO     cache dir = cache

## 2. Data generation

### 2a. Synthetic hacker-forum posts (default)

To swap in real data, replace the next cell with:
```python
from etg.data.real_forum_loader import RealForum
forum_loader = RealForum(path='data/real_posts.jsonl')
```

In [2]:
from etg.data.synthetic_forum import SyntheticForum
from etg.data.synthetic_ev import SyntheticEV

forum_loader = SyntheticForum(config)
ev_loader = SyntheticEV(config)

posts = list(forum_loader)
ev_pairs = list(ev_loader)
gt_shift_words = forum_loader.ground_truth_shift_words()

print(f'posts:          {len(posts):>6,}')
print(f'EV pairs:       {len(ev_pairs):>6,} ({sum(p.label==1 for p in ev_pairs):,} positive)')
print(f'shift words:    {sorted(gt_shift_words)}')
print()
print('sample post:        ', posts[0].text[:120])
print('sample EV pair:\n  E:', ev_pairs[0].exploit_text[:120])
print('  V:', ev_pairs[0].vulnerability_text[:120])

posts:           2,000
EV pairs:        6,000 (1,200 positive)
shift words:    ['apache', 'escalate', 'fuzz', 'kibana', 'leak', 'rce', 'root', 'samba', 'struts', 'toctou', 'tomcat', 'xss']

sample post:         for sqli ready exploit triggers lol confluence fuzz on ready ship tho bypass nope exchange module leak to
sample EV pair:
  E: chain sqli with auth bypass on joomla
  V: CVE-2024-21637: sqli vulnerability affecting joomla legacy


## 3. RT1 — Diachronic Linguistics for Exploit Trend Detection

### 3a. Build per-spell Exploit Text Graphs (ETGs)

In [3]:
from etg.data.time_spells import TimeSpellIndex
from etg.rt1.vocab import build_vocab
from etg.rt1.etg_builder import build_etg_sequence, group_posts_by_spell
from etg.rt1.laplacian_pe import attach_lap_pe

vocab = build_vocab(posts, config)
time_spells = TimeSpellIndex.from_posts(posts, config.n_spells)
posts_by_spell = group_posts_by_spell(posts, time_spells)
snapshots = build_etg_sequence(posts, posts_by_spell, vocab, config)
attach_lap_pe(snapshots, config)

print(f'vocab size:        {vocab.size:>6,}')
print(f'spells:            {config.n_spells}')
for s in snapshots:
    print(f"  t={s['t']}: nodes_observed={int(s['node_mask'].sum())}, edges={s['edge_index'].shape[1]:,}")

vocab size:           196
spells:            5
  t=0: nodes_observed=130, edges=8,974
  t=1: nodes_observed=140, edges=11,069
  t=2: nodes_observed=140, edges=12,389
  t=3: nodes_observed=136, edges=13,127
  t=4: nodes_observed=141, edges=13,570


### 3b. Visualize ETG degree distributions (Fig. 1)

In [4]:
import matplotlib.pyplot as plt
from etg.viz.rt1_plots import plot_etg_degree_distributions

fig = plot_etg_degree_distributions(
    snapshots, save_path=f'{config.figure_dir}/rt1_degree_distributions.png'
)
plt.show(); plt.close(fig)

### 3c. Train the Diachronic Graph Transformer

In [5]:
from etg.rt1.dgt_model import DGT
from etg.rt1.train_dgt import extract_embeddings, train_dgt

feature_dim = snapshots[0]['x'].shape[1]
dgt = DGT(config, feature_dim).to(device)
print(f'DGT parameters: {sum(p.numel() for p in dgt.parameters()):,}')

dgt_history = train_dgt(dgt, snapshots, config, device)
dgt_embeddings = extract_embeddings(dgt, snapshots, device)
print('embedding shapes:', [tuple(e.shape) for e in dgt_embeddings])

DGT parameters: 91,201


[04/14/26 22:08:28] INFO     DGT epoch 1/6 loss=1.5573 recon=1.1399 temp=0.8348

                    INFO     DGT epoch 2/6 loss=1.0270 recon=0.7014 temp=0.6512

                    INFO     DGT epoch 3/6 loss=0.8226 recon=0.5474 temp=0.5505

                    INFO     DGT epoch 4/6 loss=0.6876 recon=0.4478 temp=0.4794

[04/14/26 22:08:29] INFO     DGT epoch 5/6 loss=0.6453 recon=0.4411 temp=0.4084

                    INFO     DGT epoch 6/6 loss=0.5578 recon=0.3920 temp=0.3314

embedding shapes: [(196, 64), (196, 64), (196, 64), (196, 64), (196, 64)]


### 3d. Detect semantic shifts and forecast with ARIMA

In [6]:
import numpy as np
from etg.rt1.shift_detection import detect_shifts_pairwise
from etg.rt1.eval_rt1 import extrinsic_forecast, intrinsic_shift_recall_at_k, mean_shift_per_pair, analogy_hit_rate
from etg.rt1.forecast_arima import rolling_forecast_eval

masks = [s['node_mask'] for s in snapshots]
shift_results = detect_shifts_pairwise(dgt_embeddings, masks, top_quantile=config.shift_top_quantile)
for r in shift_results:
    top_idx = np.argsort(-r.scores)[:10] if len(r.scores) else []
    top_words = [vocab.token(int(r.word_ids[i])) for i in top_idx]
    print(f't={r.t_prev}→{r.t_curr}  thr={r.threshold:.3f}  shifted={len(r.shifted_ids)}  top10={top_words}')

dgt_metrics = extrinsic_forecast(shift_results, config)
dgt_metrics['shift_recall@k']  = intrinsic_shift_recall_at_k(shift_results, vocab, gt_shift_words, k=config.analogy_hit_at_k * 10)
dgt_metrics['analogy_hit@k']   = analogy_hit_rate(dgt_embeddings, vocab, k=config.analogy_hit_at_k)
results['rt1']['DGT'] = dgt_metrics
dgt_metrics

t=0→1  thr=0.015  shifted=6  top10=['see', 'postgres', 'the', 'works', 'dump', 'tested', 'attached', 'ssrf', 'day', 'first']
t=1→2  thr=0.009  shifted=7  top10=['spoof', 'v1', 'for', 'ship', 'to', 'tested', 'ready', 'csrf', 'works', 'nginx']
t=2→3  thr=0.010  shifted=7  top10=['chain', 'drop', 'spoof', 'new', 'see', 'via', 'csrf', 'shell', 'for', 'the']
t=3→4  thr=0.012  shifted=6  top10=['attached', 'need', 'see', 'poc', 'new', 'chain', 'the', 'ssrf', 'first', 'csrf']


{'MAE': 0.0005243390019208575,
 'RMSE': 0.0005243390019208575,
 'MAPE': 13.750379492441626,
 'R2': 0.0,
 'shift_recall@k': 0.4166666666666667,
 'analogy_hit@k': 1.0}

### 3e. Visualize shifts, forecast, and word trajectories (Figs. 2–5)

In [7]:
from etg.viz.rt1_plots import (
    plot_shift_score_histograms,
    plot_forecast_band,
    plot_embedding_trajectories_umap,
    plot_attention_heatmap,
)

fig = plot_shift_score_histograms(shift_results, save_path=f'{config.figure_dir}/rt1_shift_hist.png')
plt.show(); plt.close(fig)

series = mean_shift_per_pair(shift_results)
preds, actuals = rolling_forecast_eval(series, min_train=max(2, min(3, len(series)-1)), order=config.arima_order)
fig = plot_forecast_band(series, preds, actuals, save_path=f'{config.figure_dir}/rt1_forecast.png')
plt.show(); plt.close(fig)

watch = sorted(gt_shift_words)[:6]
fig = plot_embedding_trajectories_umap(dgt_embeddings, vocab, watch, save_path=f'{config.figure_dir}/rt1_trajectories.png')
plt.show(); plt.close(fig)

# Attention heatmap on a small subgraph for interpretation.
from etg.rt1.dgt_model import _build_adjacency_mask
import torch
sample_words = [w for w in ('sqli','rce','apache','nginx','exploit','payload','creds','token') if w in vocab.tokens_to_id][:6]
sample_ids = [vocab.id(w) for w in sample_words]
with torch.no_grad():
    snap = snapshots[-1]
    x = snap['x'].to(device); pe = snap['lap_pe'].to(device)
    h0 = dgt.input_proj(torch.cat([x, pe], dim=-1))
    V = x.shape[0]
    adj = _build_adjacency_mask(snap['edge_index'].to(device), V, config.dgt_k_hop, device)
    sub = h0[sample_ids]
    scores = (sub @ sub.T / (sub.shape[-1] ** 0.5))
    attn = torch.softmax(scores, dim=-1).cpu().numpy()
fig = plot_attention_heatmap(attn, sample_words, save_path=f'{config.figure_dir}/rt1_attention_heatmap.png')
plt.show(); plt.close(fig)

### 3f. RT1 baselines: word2vec (aligned) and static GCN

In [8]:
from etg.baselines.rt1_baselines import train_word2vec_per_spell, word2vec_shift_series

w2v_embeddings = train_word2vec_per_spell(posts_by_spell, vocab, config)
w2v_series = word2vec_shift_series(w2v_embeddings)
w2v_preds, w2v_actuals = rolling_forecast_eval(
    w2v_series, min_train=max(2, min(3, len(w2v_series)-1)), order=config.arima_order
)
from etg.eval.metrics_forecast import all_metrics
if len(w2v_actuals):
    results['rt1']['word2vec'] = all_metrics(w2v_actuals, w2v_preds)
else:
    results['rt1']['word2vec'] = {'MAE': float('nan'),'RMSE': float('nan'),'MAPE': float('nan'),'R2': float('nan')}

# Naive last-value baseline as a minimum floor.
from etg.rt1.forecast_arima import forecast_series
naive_preds = [forecast_series(series[:k], horizon=1, min_points=1)[0] for k in range(2, len(series))]
naive_actuals = series[2:]
if len(naive_actuals):
    results['rt1']['naive_last'] = all_metrics(np.asarray(naive_actuals), np.asarray(naive_preds))

results['rt1']

{'DGT': {'MAE': 0.0005243390019208575,
  'RMSE': 0.0005243390019208575,
  'MAPE': 13.750379492441626,
  'R2': 0.0,
  'shift_recall@k': 0.4166666666666667,
  'analogy_hit@k': 1.0},
 'word2vec': {'MAE': 0.010005343612585853,
  'RMSE': 0.010005343612585853,
  'MAPE': 249.86082375717226,
  'R2': 0.0},
 'naive_last': {'MAE': 0.0003472365049258066,
  'RMSE': 0.00038979287422292845,
  'MAPE': 9.461651921724618,
  'R2': -1.210587179625902}}

### 3g. RT1 summary bar chart

In [9]:
from etg.viz.rt1_plots import plot_rt1_summary_bar
fig = plot_rt1_summary_bar(results['rt1'], metric='MAE', save_path=f'{config.figure_dir}/rt1_summary_mae.png')
plt.show(); plt.close(fig)
fig = plot_rt1_summary_bar(results['rt1'], metric='RMSE', save_path=f'{config.figure_dir}/rt1_summary_rmse.png')
plt.show(); plt.close(fig)

## 4. RT2 — Exploit-Vulnerability Self-Supervised Linker

### 4a. CTE pretraining (masked + contrastive)

In [10]:
from etg.rt2.cte_model import CTE
from etg.rt2.pretrain_cte import pretrain_cte

cte = CTE(config).to(device)
print(f'CTE parameters: {sum(p.numel() for p in cte.parameters()):,}')
pre_hist = pretrain_cte(cte, ev_pairs, config, device)
print('final pretrain losses:', {k: round(v[-1], 4) for k, v in pre_hist.items()})

CTE parameters: 2,307,648


[04/14/26 22:08:51] INFO     CTE pretrain epoch 1/4 loss=5.2935 mlm=4.3135 cont=0.9800

[04/14/26 22:09:05] INFO     CTE pretrain epoch 2/4 loss=1.9225 mlm=1.5394 cont=0.3831

[04/14/26 22:09:19] INFO     CTE pretrain epoch 3/4 loss=1.5113 mlm=1.2086 cont=0.3027

[04/14/26 22:09:34] INFO     CTE pretrain epoch 4/4 loss=1.4190 mlm=1.1464 cont=0.2727

final pretrain losses: {'loss': 1.419, 'mlm': 1.1464, 'contrastive': 0.2727}


### 4b. CTE supervised fine-tuning

In [11]:
from etg.rt2.finetune_cte import finetune_cte
ft_hist = finetune_cte(cte, ev_pairs, config, device)
print('final fine-tune loss:', round(ft_hist['loss'][-1], 4))

[04/14/26 22:09:34] INFO     CTE fine-tune epoch 1/4 loss=0.0183

[04/14/26 22:09:35] INFO     CTE fine-tune epoch 2/4 loss=0.0046

                    INFO     CTE fine-tune epoch 3/4 loss=0.0040

[04/14/26 22:09:36] INFO     CTE fine-tune epoch 4/4 loss=0.0025

final fine-tune loss: 0.0025


### 4c. Rank exploits and compute HR/MRR/NDCG/MAP

In [12]:
from etg.rt2.ranking import rank_exploits, sample_candidate_pool
from etg.rt2.eval_rt2 import evaluate_ranking

positive_pairs = [p for p in ev_pairs if p.label == 1]
pool = sample_candidate_pool(ev_pairs, n_candidates=max(100, len(positive_pairs)//2), seed=config.seed)
scores_cte, pos_idx = rank_exploits(cte, positive_pairs, pool, config, device)
results['rt2']['CTE'] = evaluate_ranking(scores_cte, pos_idx, k=config.eval_top_k)
results['rt2']['CTE']

{'HR@10': 0.9166666666666666,
 'MRR@10': 0.4068138227513228,
 'NDCG@10': 0.5281538898358408,
 'MAP': 0.4122203211169775,
 'median_rank': 3.0,
 'mean_rank': 4.911666666666667}

### 4d. RT2 baselines (TF-IDF / BM25 / LSA / SimCSE / Contriever)

In [13]:
from etg.baselines.rt2_baselines import (
    build_tfidf_ranker, tfidf_score, lsa_score, bm25_score_matrix,
    simcse_score, contriever_score,
)
from etg.eval.metrics_ir import ranks_from_scores, all_ranking_metrics

# Reuse the same candidate pool the CTE saw: build it from positive_pairs + pool.
exploit_texts = [p.exploit_text for p in positive_pairs]
seen = {}
for t in [p.vulnerability_text for p in positive_pairs]:
    seen.setdefault(t, len(seen))
for t in pool:
    seen.setdefault(t, len(seen))
candidate_texts = list(seen.keys())
positive_indices = np.array([seen[p.vulnerability_text] for p in positive_pairs])

vec = build_tfidf_ranker(ev_pairs)
tfidf_scores = tfidf_score(vec, exploit_texts, candidate_texts)
lsa_scores = lsa_score(vec, exploit_texts, candidate_texts, dim=min(64, max(2, len(candidate_texts) - 1)))
bm25_scores = bm25_score_matrix(exploit_texts, candidate_texts)

results['rt2']['TF-IDF'] = all_ranking_metrics(ranks_from_scores(tfidf_scores, positive_indices), k=config.eval_top_k)
results['rt2']['LSA']    = all_ranking_metrics(ranks_from_scores(lsa_scores,    positive_indices), k=config.eval_top_k)
results['rt2']['BM25']   = all_ranking_metrics(ranks_from_scores(bm25_scores,   positive_indices), k=config.eval_top_k)

# Optional pretrained sentence-transformer baselines (gated by RUN_PRETRAINED_BASELINES).
if RUN_PRETRAINED_BASELINES:
    for name, fn in [('SimCSE', simcse_score), ('Contriever', contriever_score)]:
        try:
            s = fn(exploit_texts, candidate_texts)
        except Exception as e:
            log.warning(f'{name} baseline unavailable: {e}')
            s = None
        if s is None:
            results['rt2'][name] = {f'HR@{config.eval_top_k}': float('nan'), f'MRR@{config.eval_top_k}': float('nan'),
                                    f'NDCG@{config.eval_top_k}': float('nan'), 'MAP': float('nan')}
        else:
            results['rt2'][name] = all_ranking_metrics(ranks_from_scores(s, positive_indices), k=config.eval_top_k)
else:
    log.info('skipping SimCSE/Contriever (set RUN_PRETRAINED_BASELINES=True to include)')

results['rt2']

[04/14/26 22:09:37] INFO     skipping SimCSE/Contriever (set RUN_PRETRAINED_BASELINES=True to include)

{'CTE': {'HR@10': 0.9166666666666666,
  'MRR@10': 0.4068138227513228,
  'NDCG@10': 0.5281538898358408,
  'MAP': 0.4122203211169775,
  'median_rank': 3.0,
  'mean_rank': 4.911666666666667},
 'TF-IDF': {'HR@10': 0.8425,
  'MRR@10': 0.31325661375661373,
  'NDCG@10': 0.4377103163662912,
  'MAP': 0.32293873931241746},
 'LSA': {'HR@10': 0.8516666666666667,
  'MRR@10': 0.2999328703703704,
  'NDCG@10': 0.4287580081390225,
  'MAP': 0.30889159698900176},
 'BM25': {'HR@10': 0.8466666666666667,
  'MRR@10': 0.29639847883597886,
  'NDCG@10': 0.42487157263027475,
  'MAP': 0.3057266205465563}}

### 4e. Visualize CTE behavior (Figs. 6–10)

In [14]:
from etg.viz.rt2_plots import (
    plot_contrastive_view_umap,
    plot_score_histogram,
    plot_pr_roc,
    plot_cte_attention_heatmap,
    plot_rt2_summary_bar,
)
from etg.rt2.augmentations import contrastive_views
from etg.rt2.trigram_hasher import tokenize_cte
import random, torch

# (6) contrastive view UMAP
rng = random.Random(0)
sample_texts = [p.exploit_text for p in positive_pairs[:150]]
a_ids = torch.tensor([tokenize_cte(t, config.trigram_buckets, config.cte_max_len) for t in sample_texts], device=device)
b_ids = torch.tensor([tokenize_cte(contrastive_views(t, rng), config.trigram_buckets, config.cte_max_len) for t in sample_texts], device=device)
with torch.no_grad():
    z_a = cte.forward_contrastive(a_ids).cpu().numpy()
    z_b = cte.forward_contrastive(b_ids).cpu().numpy()
fig = plot_contrastive_view_umap(z_a, z_b, save_path=f'{config.figure_dir}/rt2_contrastive_umap.png')
plt.show(); plt.close(fig)

# (7) score histogram
fig = plot_score_histogram(scores_cte, pos_idx, save_path=f'{config.figure_dir}/rt2_score_hist.png')
plt.show(); plt.close(fig)

# (8) PR / ROC
fig = plot_pr_roc(scores_cte, pos_idx, save_path=f'{config.figure_dir}/rt2_pr_roc.png')
plt.show(); plt.close(fig)

# (9) CTE local-guided global attention on a sample exploit
import re
sample_exp = positive_pairs[0].exploit_text
toks = re.findall(r'[A-Za-z][A-Za-z0-9_\-]+', sample_exp)
with torch.no_grad():
    ids = torch.tensor([tokenize_cte(sample_exp, config.trigram_buckets, config.cte_max_len)], device=device)
    refined, sent, mask = cte.encode_tokens(ids)
    # Recompute the attention weights used by LG-GCA for display.
    pooled = (refined * mask.float().unsqueeze(-1)).sum(dim=1) / mask.float().sum(dim=1).clamp(min=1).unsqueeze(-1)
    Wh_g = pooled @ cte.lg_attn.W
    scores_local = torch.einsum('bld,bd->bl', refined, Wh_g) / (refined.shape[-1] ** 0.5)
    scores_local = scores_local.masked_fill(~mask, float('-inf'))
    attn_local = torch.softmax(scores_local, dim=-1).cpu().numpy()[0]
label_tokens = ['<CLS>'] + toks
label_tokens = label_tokens[: mask.sum().item()]
attn_local = attn_local[: len(label_tokens)]
fig = plot_cte_attention_heatmap(attn_local, label_tokens, save_path=f'{config.figure_dir}/rt2_cte_attention.png')
plt.show(); plt.close(fig)

# (10) RT2 summary bar
fig = plot_rt2_summary_bar(results['rt2'], metric=f'HR@{config.eval_top_k}', save_path=f'{config.figure_dir}/rt2_summary_hr.png')
plt.show(); plt.close(fig)
fig = plot_rt2_summary_bar(results['rt2'], metric=f'NDCG@{config.eval_top_k}', save_path=f'{config.figure_dir}/rt2_summary_ndcg.png')
plt.show(); plt.close(fig)

## 5. Summary

In [15]:
import pandas as pd
rt1_df = pd.DataFrame(results['rt1']).T
rt2_df = pd.DataFrame(results['rt2']).T
print('\nRT1 — diachronic forecasting')
print(rt1_df.to_string())
print('\nRT2 — exploit-vulnerability ranking')
print(rt2_df.to_string())

with open(config.results_path, 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f'\nresults saved → {config.results_path}')
print(f'figures in    → {config.figure_dir}/')


RT1 — diachronic forecasting
                 MAE      RMSE        MAPE        R2  shift_recall@k  analogy_hit@k
DGT         0.000524  0.000524   13.750379  0.000000        0.416667            1.0
word2vec    0.010005  0.010005  249.860824  0.000000             NaN            NaN
naive_last  0.000347  0.000390    9.461652 -1.210587             NaN            NaN

RT2 — exploit-vulnerability ranking
           HR@10    MRR@10   NDCG@10       MAP  median_rank  mean_rank
CTE     0.916667  0.406814  0.528154  0.412220          3.0   4.911667
TF-IDF  0.842500  0.313257  0.437710  0.322939          NaN        NaN
LSA     0.851667  0.299933  0.428758  0.308892          NaN        NaN
BM25    0.846667  0.296398  0.424872  0.305727          NaN        NaN

results saved → cache/results.json
figures in    → cache/figures/
